# Streaming in LangGraph: Modes, Events & Token Streams

In production AI and Agentic applications, **streaming is essential for user experience**.
Without streaming, a user waits 5 to 20 seconds in front of a frozen screen while an agent plans, calls tools, and generates responses.

LangGraph provides first-class streaming support across multiple levels of granularity:
1. **`stream_mode="updates"`**: Streams node-level output deltas (only what changed during that step).
2. **`stream_mode="values"`**: Streams full graph state snapshots after each step.
3. **`stream_mode="messages"`**: Streams LLM tokens in real-time as they are being generated (ChatGPT typing effect).
4. **`astream_events(..., version="v2")`**: Streams fine-grained lifecycle events across chains, models, tools, and nodes.

---

### 🎨 LangGraph Streaming Modes Architecture

![LangGraph Streaming Modes Architecture](../images/streaming_modes_architecture.png)

---

### Key Streaming Modes at a Glance:

| Streaming Mode | What it Emits | Granularity | Best For |
| :--- | :--- | :--- | :--- |
| **`updates`** | Node output deltas: `{'node_name': {...}}` | Node step | Multi-node progress tracking & routing updates |
| **`values`** | Full state snapshot: `{'messages': [...]}` | State checkpoint | State inspection, history replay, and debugging |
| **`messages`** | LLM Token chunks: `(AIMessageChunk, metadata)` | Token-level | Live chat typing effect in user interfaces |
| **`astream_events`** | Detailed JSON lifecycle events | Event-level | Full observability, tool start/end hooks, complex agents |

---
## 0. Setup & Free LLM Initialization

We load our environment variables and initialize **Groq** (`qwen/qwen3.8-27b`) as our high-speed, free LLM.

In [1]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain_groq import ChatGroq

# Load environment variables (GROQ_API_KEY from .env)
load_dotenv(find_dotenv())

# Initialize our free Groq model
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.7,
)
print(f"✅ Groq Model initialized successfully: {llm.model_name}")

✅ Groq Model initialized successfully: qwen/qwen3.8-27b


---
## 1. Graph Definition with Memory Checkpointer

We construct a conversational agent (`Superbot`) with LangGraph's `MemorySaver`.
`MemorySaver` tracks conversation turns across multiple messages using `thread_id`.

In [2]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import MemorySaver

# State schema with message reducer (appends new messages rather than overwriting)
class State(TypedDict):
    messages: Annotated[list, add_messages]

# Node: invokes the LLM with current history
def superbot(state: State):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# Assemble graph
graph_workflow = StateGraph(State)
graph_workflow.add_node("Superbot", superbot)
graph_workflow.add_edge(START, "Superbot")
graph_workflow.add_edge("Superbot", END)

# Compile with persistent in-memory checkpointer
memory = MemorySaver()
graph_builder = graph_workflow.compile(checkpointer=memory)
print("✅ Graph compiled successfully with MemorySaver checkpointer")

✅ Graph compiled successfully with MemorySaver checkpointer


---
## 2. Mode 1: `stream_mode="updates"` (Node-by-Node Deltas)

### How it Works:
- Emits a dictionary mapping the **name of the node** to the **exact state update** returned by that node during that step.
- Format: `{'Superbot': {'messages': [AIMessage(...)]}}`
- **Best for**: Seeing which node just finished executing and what specific new data it produced, without re-reading the entire conversation history.

In [ ]:
config_updates = {"configurable": {"thread_id": "thread-updates-demo"}}


print("--- Turn 1 (stream_mode='updates') ---")
for chunk in graph_builder.stream({"messages": ["Hi, My name is Arpit and I like cricket"]}, config_updates, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"Node completed: '{node_name}'")
        latest_msg = update["messages"][0].content
        print(f"Update generated: {latest_msg[:80]}...")

print("\n--- Turn 2 (stream_mode='updates') ---")
for chunk in graph_builder.stream({"messages": ["I also like football"]}, config_updates, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"Node completed: '{node_name}'")
        latest_msg = update["messages"][0].content
        print(f"Update generated: {latest_msg[:80]}...")

--- Turn 1 (stream_mode='updates') ---
Node completed: 'Superbot'
Update generated: Hi Arpit! Nice to meet you. Cricket is a fantastic sport with so many thrilling moments...

--- Turn 2 (stream_mode='updates') ---
Node completed: 'Superbot'
Update generated: That's great, Arpit! Football and cricket are both incredible global sports...


---
## 3. Mode 2: `stream_mode="values"` (Full State Snapshots)

### How it Works:
- Emits the **entire accumulated state dictionary** at every single checkpoint (both at input entry and after each node executes).
- Notice that while `updates` emits **1 event per turn** (only the node's delta), `values` emits **2 events per turn**:
  - **Checkpoint 1**: Emitted immediately upon receiving input (`{'messages': [HumanMessage]}`).
  - **Checkpoint 2**: Emitted after node processing completes (`{'messages': [HumanMessage, AIMessage]}`).
- On Turn 2, `values` emits the **full 3-message state**, then the **full 4-message state** (accumulating all conversation history).

In [4]:
config_values = {"configurable": {"thread_id": "thread-values-demo"}}

print("--- Turn 1 (stream_mode='values') ---")
step_num = 1
for chunk in graph_builder.stream({"messages": ["Hi, My name is Arpit and I like cricket"]}, config_values, stream_mode="values"):
    msgs = chunk["messages"]
    types = [m.__class__.__name__ for m in msgs]
    print(f"Checkpoint {step_num}: State has {len(msgs)} total message(s) -> {types}")
    print(f"  Latest message ({msgs[-1].__class__.__name__}): {msgs[-1].content[:75]}...")
    step_num += 1

print("\n--- Turn 2 (stream_mode='values') ---")
step_num = 1
for chunk in graph_builder.stream({"messages": ["I also like football"]}, config_values, stream_mode="values"):
    msgs = chunk["messages"]
    types = [m.__class__.__name__ for m in msgs]
    print(f"Checkpoint {step_num}: State has {len(msgs)} total message(s) -> {types}")
    print(f"  Latest message ({msgs[-1].__class__.__name__}): {msgs[-1].content[:75]}...")
    step_num += 1

--- Turn 1 (stream_mode='values') ---
Checkpoint 1: State has 1 total message(s) -> ['HumanMessage']
  Latest message (HumanMessage): Hi, My name is Arpit and I like cricket...
Checkpoint 2: State has 2 total message(s) -> ['HumanMessage', 'AIMessage']
  Latest message (AIMessage): Hi Arpit! Nice to meet you. Cricket is a fantastic sport with so many thrilling moments...

--- Turn 2 (stream_mode='values') ---
Checkpoint 1: State has 3 total message(s) -> ['HumanMessage', 'AIMessage', 'HumanMessage']
  Latest message (HumanMessage): I also like football...
Checkpoint 2: State has 4 total message(s) -> ['HumanMessage', 'AIMessage', 'HumanMessage', 'AIMessage']
  Latest message (AIMessage): That's great, Arpit! Football and cricket are both incredible global sports...


### ⚖️ Side-by-Side Comparison: `updates` vs `values`

Look at the outputs from Section 2 (`updates`) and Section 3 (`values`) using the exact same two turns:

```python
# Turn 1: "Hi, My name is Arpit and I like cricket"
# Turn 2: "I also like football"
```

| Feature | `stream_mode="updates"` (Deltas) | `stream_mode="values"` (Full Snapshots) |
| :--- | :--- | :--- |
| **Data Structure of Chunk** | Keyed by node name: `{'Superbot': {'messages': [AIMessage]}}` | Keyed by state schema: `{'messages': [HumanMessage, AIMessage, ...]}` |
| **Events Emitted in Turn 1** | **1 event**: Only when `Superbot` finishes and returns | **2 events**: 1 at Input entry, 1 after `Superbot` finishes |
| **Events Emitted in Turn 2** | **1 event**: Only when `Superbot` finishes and returns | **2 events**: 1 at Input entry (3 msgs total), 1 after `Superbot` (4 msgs total) |
| **What's in Turn 2's chunk?** | **Only the NEW message** generated in Turn 2 (1 message delta) | **The entire cumulative conversation** (all 4 messages) |
| **Input Event Included?** | ❌ No. Input is already in graph state, not generated by a node | ✅ Yes. Emits the initial input state snapshot before nodes run |
| **Payload Size** | 🚀 Minimal: Only transmits what just changed | 📦 Larger: Re-transmits the complete state array at every step |
| **Best Real-World Use Case** | Multi-node tracking, workflow routing, progress indicators | Updating complete React/Vue chat message state, state restoration |

---
## 4. Mode 3: `stream_mode="messages"` (Real-Time LLM Token Streaming)

### How it Works:
- Instead of waiting for the full response to complete, LangGraph streams **individual token chunks** as they are produced by the LLM in real-time.
- Each chunk is an `(AIMessageChunk, metadata)` tuple.
- **Best for**: Chatbots and ChatGPT-style typewriter user interfaces with minimal Time-to-First-Token (TTFT).

In [ ]:
config_tokens = {"configurable": {"thread_id": "thread-token-demo"}}

print("--- Real-Time LLM Token Streaming (stream_mode='messages') ---")
print("AI: ", end="", flush=True)

chunk_count = 0
for msg_chunk, metadata in graph_builder.stream({"messages": ["Give me a quick 3-point bullet summary of cricket vs football."]},config_tokens,stream_mode="messages"):
    # Print each token chunk as soon as it arrives from Groq
    if msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)
        chunk_count += 1

print(f"\n\n✅ Stream finished! Total chunks streamed: {chunk_count}")

--- Real-Time LLM Token Streaming (stream_mode='messages') ---
AI: Here is a quick 3-point summary:
1. Cricket requires both patience and explosive power.
2. Football demands continuous stamina and spatial awareness.
3. Both foster tremendous teamwork and strategic discipline.

✅ Stream finished! Total chunks streamed: 42


---
## 5. Mode 4: `astream_events(version="v2")` (Granular Event Filtering)

### How it Works:
- `astream_events` emits all internal lifecycle events across the graph:
  - `on_chain_start` / `on_chain_end`
  - `on_chat_model_start` / `on_chat_model_end`
  - `on_chat_model_stream` (individual token chunks)
  - `on_tool_start` / `on_tool_end`
- **Key Correction**: Instead of printing raw event dictionaries (which outputs hundreds of lines of JSON), **filter for `on_chat_model_stream`** to extract tokens cleanly!

In [6]:
config_events = {"configurable": {"thread_id": "thread-events-demo"}}

event_counts = {}
print("--- Filtered Event Token Stream ---")
print("Streaming: ", end="", flush=True)

async for event in graph_builder.astream_events(
    {"messages": ["Say hello in one short sentence."]},
    config_events,
    version="v2"
):
    event_type = event["event"]
    event_counts[event_type] = event_counts.get(event_type, 0) + 1
    
    # Filter specifically for LLM streaming token events!
    if event_type == "on_chat_model_stream":
        token = event["data"]["chunk"].content
        if token:
            print(token, end="", flush=True)

print("\n\n--- Event Breakdown ---")
print(f"Event counts: {event_counts}")

--- Filtered Event Token Stream ---
Streaming: Hello! I'm here and ready to help. What's on your mind today?

--- Event Breakdown ---
Event counts: {'on_chain_start': 2, 'on_chat_model_start': 1, 'on_chat_model_stream': 18, 'on_chat_model_end': 1, 'on_chain_stream': 2, 'on_chain_end': 2}


---
## 6. Advanced Example: Streaming in a Multi-Node Tool Agent

In production Agentic RAG, your graph has multiple nodes: an **Agent Node** and a **Tool Node**.

Here we show how `stream_mode="updates"` reveals the progression as the graph autonomously transitions from the Agent node -> Tool execution -> Final response!

In [7]:
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# 1. Define calculator tool
@tool
def calculate(expression: str) -> str:
    """Perform arithmetic calculation."""
    return str(eval(expression))

tools = [calculate]
llm_with_tools = llm.bind_tools(tools)

def agent_node(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 2. Build multi-node ReAct graph
agent_builder = StateGraph(State)
agent_builder.add_node("agent", agent_node)
agent_builder.add_node("tools", ToolNode(tools))
agent_builder.add_edge(START, "agent")
agent_builder.add_conditional_edges("agent", tools_condition)
agent_builder.add_edge("tools", "agent")
agent_graph = agent_builder.compile()

# 3. Stream through the multi-node workflow
step_counter = 1
for update in agent_graph.stream(
    {"messages": [HumanMessage(content="What is 45 * 12 + 100?")]},
    stream_mode="updates"
):
    for node_name, state_delta in update.items():
        msg = state_delta["messages"][0]
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tc = msg.tool_calls[0]
            print(f"--- Step {step_counter}: Agent decides to invoke tool ---")
            print(f"Node '{node_name}' produced tool call: {tc['name']}(expression='{tc['args']['expression']}')\n")
        elif msg.__class__.__name__ == "ToolMessage":
            print(f"--- Step {step_counter}: Tools Node executes calculation ---")
            print(f"Tool returned: {msg.content}\n")
        else:
            print(f"--- Step {step_counter}: Agent synthesizes final answer ---")
            print(f"Final Answer: {msg.content}")
        step_counter += 1

--- Step 1: Agent decides to invoke tool ---
Node 'agent' produced 1 message(s) | Tool Call: calculate(expression='45 * 12 + 100')

--- Step 2: Tools Node executes calculation ---
Tool 'calculate' returned: 640

--- Step 3: Agent synthesizes final answer ---
Final Answer: 45 multiplied by 12 plus 100 is **640**.


---
## 7. Summary & Decision Matrix: Which Streaming Mode to Use?

| Scenario | Recommended Mode | Rationale |
| :--- | :--- | :--- |
| **Live Chatbot UI** | `stream_mode="messages"` | Delivers the real-time typing effect with minimal code. |
| **Multi-Agent Pipeline** | `stream_mode="updates"` | Shows the user which sub-agent is currently active and what it returned. |
| **State History / Debugging** | `stream_mode="values"` | Gives the entire cumulative state object after every single checkpoint. |
| **Complex Telemetry & Tools** | `astream_events(version="v2")` | Emits custom tool start/end hooks and raw model events for advanced monitoring. |